# 남한 지도 생성기 v3 (레고 픽셀아트 소스용)

- **북한 제외**, 남한만 표시
- **도·특별시·광역시 경계만** (시군구 경계 제거)
- **컬러**: 배경 흰색, 육지 흰색, 경계선 검은색 (흑백 윤곽선 지도)
- **제주도**를 육지와 가깝게 배치
- **가로 픽셀 수(`PX_W`)** 로 출력 크기 조정


In [ ]:
# ── 설정 ──

# 파일 경로
DATA = "skorea-municipalities-2018-geo.json"
OUT_PLAIN = "south_korea_plain.png"   # 1) 격자 없는 일반 지도
OUT_GRID  = "south_korea_grid.png"    # 2) 격자 있는 고해상도 지도

# 출력 가로 픽셀 수 (16의 배수 권장, 세로는 데이터 비율에 맞춰 자동 계산)
# 1) 격자 없는 일반 지도용 (픽셀아트 소스)
PX_W = 256

# 2) 격자 있는 고해상도 지도용 (출력해서 보면서 만들 용도)
#    크면 클수록 격자칸이 커져서 보기 편함
#    1024 → 격자칸 64px(약 1.6cm), 2048 → 128px(약 3.3cm), 4096 → 256px(약 6.5cm)
PX_W_GRID = 2048

# 모눈종이 격자
GRID_SIZE  = 16          # 격자 한 칸 크기 (픽셀)
GRID_MAJOR = 16          # 몇 칸마다 굵은 선을 그릴지 (16 → 16×16=256px마다 굵은 선)
GRID_MINOR_COLOR = "#CCCCCC"  # 얇은 격자선
GRID_MAJOR_COLOR = "#888888"  # 굵은 격자선
GRID_MINOR_LW = 0.3            # 얇은 선 두께
GRID_MAJOR_LW = 0.8            # 굵은 선 두께

# 섬 필터 (km² 단위)
# 200 km² 이하 섬 제거 (거문도 제거, 제주도 유지)
ISLAND_MIN_AREA_KM2 = 200
SQDEG_TO_KM2 = 9845

# 색상 (흑백 윤곽선)
BG_COLOR   = "#FFFFFF"   # 배경 (흰색)
LAND_COLOR = "#FFFFFF"   # 육지 (흰색)
EDGE_COLOR = "#000000"   # 시도 경계 (검은색)

# 시도 경계선 두께 (출력 해상도에 비례, 함수로 자동 계산)
def edge_lw(px_w):
    return max(1, px_w // 200)  # 256px → 1px, 2048px → 10px

# PX_W, PX_W_GRID를 격자 크기의 배수로 정렬
PX_W = (PX_W // GRID_SIZE) * GRID_SIZE
PX_W_GRID = (PX_W_GRID // GRID_SIZE) * GRID_SIZE

# 제주도 위치 보정 (위도 방향 북쪽 이동, degree 단위)
# 0 = 원위치, 0.3 ≈ 33km, 0.5 ≈ 55km, 0.7 ≈ 77km
JEJU_SHIFT = 0.5

# 해안선 단순화 (degree 단위, 0=해제)
# 0.005 ≈ 550m (픽셀아트용 추천), 0.01 ≈ 1.1km (강하게)
COAST_SMOOTH = 0.005

# 시도 코드 → 이름
SIDO_NAMES = {
    "11": "서울특별시", "21": "부산광역시", "22": "대구광역시", "23": "인천광역시",
    "24": "광주광역시", "25": "대전광역시", "26": "울산광역시", "29": "세종특별자치시",
    "31": "경기도", "32": "강원도", "33": "충청북도", "34": "충청남도",
    "35": "전라북도", "36": "전라남도", "37": "경상북도", "38": "경상남도",
    "39": "제주특별자치도",
}


In [ ]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from shapely.geometry import shape
from shapely.ops import unary_union
from shapely.affinity import translate


def filter_islands(geom, min_area):
    """MultiPolygon/Polygon에서 min_area 미만 폴리곤(섬) 제거"""
    if geom.is_empty:
        return geom
    if geom.geom_type == "Polygon":
        return geom if geom.area >= min_area else None
    polys = [p for p in geom.geoms if p.area >= min_area]
    return unary_union(polys) if polys else None


def get_polys(geom):
    """Polygon/MultiPolygon → list[Polygon]"""
    if geom.is_empty:
        return []
    if geom.geom_type == "Polygon":
        return [geom]
    return list(geom.geoms)


# ── 1. 데이터 로드 (남한만) ──
with open(DATA, encoding="utf-8") as f:
    gj = json.load(f)

sido_groups = {}  # 시도별 폴리곤
for ft in gj["features"]:
    name = ft["properties"]["name"]
    code = ft["properties"]["code"]
    if name == "울릉군":
        continue  # 울릉도/독도 제외
    geom = shape(ft["geometry"])
    if code[:2] == "39" and JEJU_SHIFT > 0:
        geom = translate(geom, yoff=JEJU_SHIFT)
    sido_groups.setdefault(code[:2], []).append(geom)

# ── 2. 병합 + 섬 필터 ──
min_area_sqdeg = ISLAND_MIN_AREA_KM2 / SQDEG_TO_KM2
sido_polys = {}
for code, geoms in sido_groups.items():
    merged = unary_union(geoms)
    filtered = filter_islands(merged, min_area_sqdeg)
    if filtered is not None:
        sido_polys[code] = filtered

all_land = unary_union(list(sido_polys.values()))

# ── 3. 해안선 단순화 ──
if COAST_SMOOTH > 0:
    sido_polys = {k: v.simplify(COAST_SMOOTH, preserve_topology=False)
                  for k, v in sido_polys.items()}
    all_land = all_land.simplify(COAST_SMOOTH, preserve_topology=False)
    print(f"해안선 단순화: tolerance={COAST_SMOOTH}°")

print(f"시도: {len(sido_polys)}개")


In [ ]:
# ── 4. 전체 범위 계산 ──
xs, ys = [], []
for g in sido_polys.values():
    minx, miny, maxx, maxy = g.bounds
    xs += [minx, maxx]
    ys += [miny, maxy]

xmin, xmax = min(xs), max(xs)
ymin, ymax = min(ys), max(ys)
xpad = (xmax - xmin) * 0.02  # 동쪽 여백 (포항)
xmax += xpad

# ── 5. 공통 함수 ──
data_aspect = 1.391 / 1.127  # 경도/위도 보정
xrange = xmax - xmin
yrange = ymax - ymin
fig_aspect = (xrange * data_aspect) / yrange  # 가로/세로 비율


def draw_map(px_w, out_path, draw_grid=False):
    """지도를 그려서 out_path에 저장"""
    if draw_grid:
        # 격자용: PX_H를 256의 배수로 올림 (16x16 블록 완전 보장)
        major_px = GRID_SIZE * GRID_MAJOR  # 256
        px_h = int(px_w / fig_aspect)
        px_h = ((px_h + major_px - 1) // major_px) * major_px
        # y 범위 확장: px_h에 맞춰 ymax를 올려서 완전한 블록 확보
        yrange_new = (px_h / px_w) * data_aspect * (xmax - xmin)
        ymax_draw = ymin + yrange_new  # ymin 고정, ymax 확장
    else:
        # 일반용: 원래 데이터 비율 그대로
        px_h = int(px_w / fig_aspect)
        px_h = (px_h // GRID_SIZE) * GRID_SIZE  # 16의 배수 정렬
        ymax_draw = ymax  # y 범위 확장 없음

    DPI = 100
    w_inch = px_w / DPI
    h_inch = px_h / DPI

    fig, ax = plt.subplots(figsize=(w_inch, h_inch), dpi=DPI)
    ax.set_aspect(data_aspect)
    ax.set_facecolor(BG_COLOR)
    fig.set_facecolor(BG_COLOR)

    # 육지 fill (흰색, 틈 없음)
    for poly in get_polys(all_land):
        x, y = poly.exterior.xy
        ax.fill(x, y, facecolor=LAND_COLOR, edgecolor="none", zorder=1)

    # 시도 경계선 (검은색)
    lw = edge_lw(px_w)
    for g in sido_polys.values():
        for poly in get_polys(g):
            x, y = poly.exterior.xy
            ax.plot(x, y, color=EDGE_COLOR, linewidth=lw, zorder=2)

    # 모눈종이 격자
    if draw_grid:
        # 격자선 두께도 해상도에 비례
        scale = px_w / 256  # 256px 기준에서 배율
        minor_lw = GRID_MINOR_LW * scale
        major_lw = GRID_MAJOR_LW * scale

        def px_to_x(px):
            return xmin + (px / px_w) * (xmax - xmin)
        def px_to_y(py):
            return ymax_draw - (py / px_h) * (ymax_draw - ymin)

        # 세로선
        for px in range(0, px_w + 1, GRID_SIZE):
            is_major = (px % (GRID_SIZE * GRID_MAJOR) == 0)
            color = GRID_MAJOR_COLOR if is_major else GRID_MINOR_COLOR
            lw_g = major_lw if is_major else minor_lw
            xv = px_to_x(px)
            ax.plot([xv, xv], [ymin, ymax_draw], color=color, linewidth=lw_g, zorder=3)

        # 가로선
        for py in range(0, px_h + 1, GRID_SIZE):
            is_major = (py % (GRID_SIZE * GRID_MAJOR) == 0)
            color = GRID_MAJOR_COLOR if is_major else GRID_MINOR_COLOR
            lw_g = major_lw if is_major else minor_lw
            yv = px_to_y(py)
            ax.plot([xmin, xmax], [yv, yv], color=color, linewidth=lw_g, zorder=3)

    # 저장
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax_draw)
    ax.axis("off")
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.savefig(out_path, dpi=DPI, facecolor=fig.get_facecolor())
    plt.close()

    from PIL import Image
    img = Image.open(out_path)
    w, h = img.size
    print(f"  saved: {out_path}")
    print(f"  크기: {w} x {h} px ({w/DPI*0.0254:.2f}m x {h/DPI*0.0254:.2f}m)")
    if draw_grid:
        cells_w = px_w // GRID_SIZE
        cells_h = px_h // GRID_SIZE
        print(f"  격자: {GRID_SIZE}px 단위, {GRID_SIZE*GRID_MAJOR}px마다 굵은 선")
        print(f"  격자 칸: {cells_w} x {cells_h} = {cells_w * cells_h}칸")
        print(f"  16x16 블록: {cells_w // GRID_MAJOR} x {cells_h // GRID_MAJOR} = {(cells_w // GRID_MAJOR) * (cells_h // GRID_MAJOR)}블록")


# ── 6. 두 개 출력 ──
print(f"시도: {len(sido_polys)}개")
print()

print("1) 격자 없는 일반 지도")
draw_map(PX_W, OUT_PLAIN, draw_grid=False)
print()

print("2) 격자 있는 고해상도 지도 (출력용)")
draw_map(PX_W_GRID, OUT_GRID, draw_grid=True)
